# Does AlphaGenome's predicted RNA-seq track real individual-level expression? (GEUVADIS validation)

`genotype_cnn_alignment_deeplift.ipynb` (and the pigmentation CNN it analyzes) treats
AlphaGenome's per-individual RNA-seq prediction as ground truth input to a downstream model, but
never checks that prediction against a real measured RNA-seq value. This notebook does that check
directly, using [GEUVADIS](https://www.ebi.ac.uk/biostudies/arrayexpress/studies/E-GEUV-1) --
RNA-seq of 462 lymphoblastoid cell lines (LCLs) derived from 1000 Genomes individuals, the same
cohort this repo's `1kg_high_coverage` dataset already has phased genotypes for.

**Why this comparison is meaningful, not just approximate.** AlphaGenome's ENCODE/GTEx-derived
RNA-seq catalog includes two tracks that are essentially the same underlying biology GEUVADIS
measured:

- `GM12878` (`EFO:0002784`, ENCODE) -- a specific, widely used reference lymphoblastoid cell line.
- `Cells_EBV-transformed_lymphocytes` (`EFO:0000572`, GTEx) -- GTEx's own LCL tissue, prepared the
  same way (EBV-transformed lymphoblastoid cell lines) as every GEUVADIS sample.

Both tracks are used below; GM12878 is the primary comparison (an ENCODE track restricted to
`polyA plus RNA-seq`, matching GEUVADIS's own poly-A selected library prep).

**Method.** For a pilot set of `N_GENES=10` genes and `N_INDIVIDUALS=30` individuals (chosen
below, not hand-picked -- see Sections 4-5 for the selection rules and why):

1. Build each individual's real diploid consensus sequence for a 524,288 bp window around each
   gene's MANE Select TSS, using this repo's existing on-demand `bcftools consensus` helper
   (`genomics.predictors.genotype_based.analysis.individual_consensus`) -- no dependency on these
   genes already being part of some predictor's materialized dataset.
2. Call the live AlphaGenome API (`predict_sequence`) on each haplotype, restricted to `RNA_SEQ`
   and the two LCL ontology terms above.
3. Reduce each haplotype's predicted track to one scalar per gene by summing the `polyA plus
   RNA-seq` signal (both strands) over the gene's MANE Select exon union -- the same read-in-exons
   quantity RPKM measures -- then average the two haplotypes for one diploid prediction per
   individual.
4. Compare that predicted scalar to GEUVADIS's own measured RPKM for the same gene/individual,
   **within each gene, across the 30 individuals** (Spearman/Pearson correlation) -- the direct
   test of whether AlphaGenome's signal moves with *real* inter-individual expression differences,
   not just with which genes are highly expressed in general.

**The caveat this notebook is designed to quantify, not just state.** AlphaGenome's own published
evaluation reports it is considerably stronger at predicting *relative* variant effects (e.g. which
allele increases or decreases expression, used for eQTL/pathogenicity classification) than at
resolving fine-grained *absolute* differences in individual-level expression magnitude. A modest
or even weak per-gene correlation here would be consistent with that -- it would say AlphaGenome's
signal is a noisy proxy for absolute inter-individual expression at this scale, which is exactly
the thing to know before trusting it as a feature source. Section 8 adds a second, cheaper check
(no extra API calls): whether the *population-mean* predicted signal at least tracks *which genes*
are more or less expressed overall -- a coarser question AlphaGenome is expected to do better on.

## Section 0 -- Setup

In [ ]:
%matplotlib inline
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/home/breno/I2CA/genomics")
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)

# individual_consensus.py shells out to bcftools/samtools; make sure they're resolvable
# regardless of how this kernel's PATH was set up (belt-and-suspenders -- the `genomics` conda
# env this kernel runs in already ships both, but plain `sys.executable`-based kernel launches
# don't always inherit that env's bin/ on PATH the way an activated shell would).
GENOMICS_ENV_BIN = "/home/breno/miniforge3/envs/genomics/bin"
if GENOMICS_ENV_BIN not in os.environ.get("PATH", ""):
    os.environ["PATH"] = GENOMICS_ENV_BIN + os.pathsep + os.environ.get("PATH", "")
assert subprocess.run(["bcftools", "--version"], capture_output=True).returncode == 0
assert subprocess.run(["samtools", "--version"], capture_output=True).returncode == 0

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scistats

from alphagenome.data import genome, gene_annotation
from alphagenome.data import transcript as transcript_utils
from alphagenome.models import dna_client

from genomics.core.data_registry import resolve_dataset
from genomics.core.reference_registry import resolve_reference
from genomics.core.raw_variants import raw_variant_chromosome_dir, kg1000_vcf_name
from genomics.predictors.genotype_based.analysis.individual_consensus import build_haplotype_fasta
from genomics.predictors.genotype_based.analysis.live_alphagenome_prediction import resolve_api_key

NOTEBOOK_CACHE_DIR = REPO_ROOT / "notebooks" / ".cache" / "alphagenome_geuvadis_validation"
NOTEBOOK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

N_GENES = 10
N_INDIVIDUALS = 30
WINDOW_SIZE = dna_client.SEQUENCE_LENGTH_500KB  # 524,288 bp, same window size the pigmentation notebook uses
RANDOM_SEED = 42

ORGANISM = dna_client.Organism.HOMO_SAPIENS
GM12878_ONTOLOGY = "EFO:0002784"   # ENCODE GM12878 -- reference lymphoblastoid cell line
GTEX_LCL_ONTOLOGY = "EFO:0000572"  # GTEx Cells_EBV-transformed_lymphocytes -- GEUVADIS's own tissue

ag_client = dna_client.create(api_key=resolve_api_key())
print("AlphaGenome client ready.")

## Section 1 -- GENCODE annotations (MANE Select transcripts, for TSS + exon structure)

Same cache location/URL `genotype_cnn_alignment_deeplift.ipynb` uses, so this reuses that
notebook's download if it already ran.

In [ ]:
import requests

GTF_CACHE_DIR = REPO_ROOT / "notebooks" / ".cache" / "annotations"
GTF_URL = (
    "https://storage.googleapis.com/alphagenome/reference/gencode/"
    "hg38/gencode.v46.annotation.gtf.gz.feather"
)


def load_gtf(cache_dir: Path) -> pd.DataFrame:
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / "gencode.v46.annotation.gtf.gz.feather"
    if not cache_path.exists():
        r = requests.get(GTF_URL, timeout=120)
        r.raise_for_status()
        cache_path.write_bytes(r.content)
    return pd.read_feather(cache_path)


gtf = load_gtf(GTF_CACHE_DIR)
gtf_mane = gene_annotation.filter_to_mane_select_transcript(gene_annotation.filter_protein_coding(gtf))
gene_id_map = gtf.drop_duplicates("gene_name").set_index("gene_name")["gene_id"]
tss_df_mane = gene_annotation.extract_tss(gtf_mane)
tss_df_mane = tss_df_mane.assign(ensembl_base=tss_df_mane["gene_id"].str.split(".").str[0])
transcript_extractor_mane = transcript_utils.TranscriptExtractor(gtf_mane)
print(f"GTF loaded: {len(gtf):,} rows -- {gtf_mane['gene_name'].nunique():,} genes with a MANE Select transcript")

## Section 2 -- GEUVADIS measured expression (ground truth)

`GD462.GeneQuantRPKM.50FN.samplename.resk10.txt.gz` is GEUVADIS's own published, per-individual
(technical replicates already collapsed), quantile-normalized gene-level RPKM matrix -- 462
individuals x ~23.7k genes -- from the GEUVADIS analysis results
(`E-GEUV-1/analysis_results/`, still served from EBI's legacy microarray FTP mirror). Restricted
to autosomes here (chrX/Y dropped) to avoid sex-linked dosage complicating the per-gene comparison.

In [ ]:
GEUVADIS_URL = (
    "https://ftp.ebi.ac.uk/pub/databases/microarray/data/experiment/GEUV/"
    "E-GEUV-1/analysis_results/GD462.GeneQuantRPKM.50FN.samplename.resk10.txt.gz"
)
GEUVADIS_CACHE_PATH = NOTEBOOK_CACHE_DIR / "GD462.GeneQuantRPKM.50FN.samplename.resk10.txt.gz"

if not GEUVADIS_CACHE_PATH.exists():
    r = requests.get(GEUVADIS_URL, timeout=180)
    r.raise_for_status()
    GEUVADIS_CACHE_PATH.write_bytes(r.content)

geuv = pd.read_csv(GEUVADIS_CACHE_PATH, sep="\t")
geuv["ensembl_base"] = geuv["TargetID"].str.split(".").str[0]
geuv = geuv[geuv["Chr"].astype(str).isin([str(i) for i in range(1, 23)])].reset_index(drop=True)
GEUV_SAMPLE_COLS = [c for c in geuv.columns if c not in ("TargetID", "Gene_Symbol", "Chr", "Coord", "ensembl_base")]
print(f"GEUVADIS: {len(geuv):,} autosomal genes x {len(GEUV_SAMPLE_COLS)} individuals")

## Section 3 -- Genotypes and reference genome (this repo's `1kg_high_coverage` dataset)

GEUVADIS individuals are a subset of the 1000 Genomes panel, so this repo's already-registered
`1kg_high_coverage` dataset (3,202 individuals, phased SNV/INDEL/SV calls) covers essentially all
of them -- no separate genotype source needed. Per-chromosome phased VCFs are resolved from the
canonical registry location first, falling back to this machine's pre-unification legacy copy
(`top3/longevity_dataset/vcf_chromosomes/`) for any chromosome the canonical location doesn't have
locally yet -- avoids re-downloading multi-GB VCFs that are already sitting on disk.

In [ ]:
DATASET_REF = resolve_dataset("1kg_high_coverage")
dataset_metadata = json.loads((DATASET_REF.path / "dataset_metadata.json").read_text())
individuals_pedigree = dataset_metadata["individuals_pedigree"]

REF_GENOME = resolve_reference()
REF_FASTA_PATH = REF_GENOME.path
assert REF_FASTA_PATH.exists(), f"Reference genome not found at {REF_FASTA_PATH}; run genomics workflows to fetch it."

CANONICAL_VCF_DIR = raw_variant_chromosome_dir("1kg_high_coverage")
FALLBACK_VCF_DIR = Path("/dados/GENOMICS_DATA/top3/longevity_dataset/vcf_chromosomes")


def resolve_chrom_vcf(chrom: str) -> Path:
    name = kg1000_vcf_name(chrom)
    for candidate_dir in (CANONICAL_VCF_DIR, FALLBACK_VCF_DIR):
        candidate = candidate_dir / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"No phased VCF for {chrom} in {CANONICAL_VCF_DIR} or {FALLBACK_VCF_DIR}. "
        "Use genomics.core.raw_variants.ensure_kg1000_vcfs to download it."
    )


geuv_individuals = set(GEUV_SAMPLE_COLS)
dataset_individuals = set(individuals_pedigree)
overlap_individuals = sorted(dataset_individuals & geuv_individuals)
print(f"{len(overlap_individuals)}/{len(geuv_individuals)} GEUVADIS individuals have genotypes in {DATASET_REF.dataset_id}")

## Section 4 -- Gene selection (data-driven, not hand-picked)

Picks `N_GENES=10` autosomal, protein-coding genes with a MANE Select transcript that are:

- **present in GEUVADIS** (joined on the stripped Ensembl gene ID),
- **actually expressed in LCLs** (population mean RPKM >= 5 -- excludes noise-dominated,
  near-silent genes where any correlation would be swamped by measurement noise on both sides),
- **fit a full 524,288 bp AlphaGenome window** centered on the TSS without running off the end of
  the chromosome,

ranked by **coefficient of variation** (population std / mean RPKM) across all 462 GEUVADIS
individuals, taking the top 10. High cross-individual variability is a prerequisite, not a nice-to-
have: a gene with (almost) constant expression across the population gives a correlation analysis
no real variation to recover, regardless of how good the underlying model is.

In [ ]:
merged_genes = tss_df_mane.merge(geuv, on="ensembl_base", how="inner")

expr = merged_genes[GEUV_SAMPLE_COLS].to_numpy(dtype=float)
mean_rpkm = expr.mean(axis=1)
std_rpkm = expr.std(axis=1)
cv_rpkm = np.divide(std_rpkm, mean_rpkm, out=np.zeros_like(std_rpkm), where=mean_rpkm > 0)
merged_genes = merged_genes.assign(mean_rpkm=mean_rpkm, cv_rpkm=cv_rpkm)

CHROM_LENGTHS = {}
for line in open(f"{REF_FASTA_PATH}.fai"):
    name, length = line.split("\t")[:2]
    CHROM_LENGTHS[name] = int(length)

half_window = WINDOW_SIZE // 2


def window_bounds(row):
    chrom = row["Chromosome"]
    tss = int(row["Start"])
    start, end = tss - half_window, tss + half_window
    in_bounds = chrom in CHROM_LENGTHS and start >= 0 and end <= CHROM_LENGTHS[chrom]
    return pd.Series({"win_start": start, "win_end": end, "win_ok": in_bounds})


merged_genes = merged_genes.join(merged_genes.apply(window_bounds, axis=1))

gene_candidates = (
    merged_genes[(merged_genes["mean_rpkm"] >= 5) & merged_genes["win_ok"]]
    .drop_duplicates("gene_name")
)
pilot_genes = (
    gene_candidates.sort_values("cv_rpkm", ascending=False)
    .head(N_GENES)[
        ["gene_name", "ensembl_base", "Chromosome", "Start", "Strand", "win_start", "win_end", "mean_rpkm", "cv_rpkm"]
    ]
    .reset_index(drop=True)
    .rename(columns={"Chromosome": "chrom", "Start": "tss", "Strand": "strand"})
)
print(f"{len(gene_candidates):,} candidate genes (mean RPKM >= 5, in-bounds window); selected top {N_GENES} by CV:")
pilot_genes

## Section 5 -- Individual selection

`N_INDIVIDUALS=30` sampled uniformly at random (fixed seed, reproducible) from the
`overlap_individuals` computed in Section 3. GEUVADIS spans 5 populations (CEU/FIN/GBR/TSI/YRI);
population membership is reported below for transparency, not enforced as a stratification
constraint -- a pilot of this size doesn't need it, and enforcing it would bias the sample toward
being more balanced than a real deployment's individual set would be.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
pilot_individuals = sorted(rng.choice(overlap_individuals, size=N_INDIVIDUALS, replace=False).tolist())

pop_counts = pd.Series([individuals_pedigree[s]["population"] for s in pilot_individuals]).value_counts()
print(f"{N_INDIVIDUALS} individuals sampled (seed={RANDOM_SEED}). Population breakdown:")
print(pop_counts.to_string())

## Section 6 -- Personalized sequences + live AlphaGenome RNA-seq predictions

**Reference window FASTA header convention.** `bcftools consensus` only applies variants from a
VCF when the FASTA record's name matches the VCF's `CHROM` in the specific `chrom:start-end`
format this repo's own `references/windows/<gene>/ref.window.fa` files already use (verified
against `HERC2`'s existing materialized window) -- a plain gene-name header silently applies
**zero** variants (bcftools warns `Sequence "<name>" not in <vcf>` and every "consensus" comes back
identical to the reference). Reproduced deliberately below rather than assumed.

**Aggregation.** Each haplotype's predicted track is reduced to one scalar by summing `polyA plus
RNA-seq` signal (`+` and `-` strand columns) over the gene's MANE Select exon union in local window
coordinates -- the same "reads landing in exons" quantity RPKM is built from. The two haplotypes'
scalars are then averaged for one diploid prediction per individual/gene. Both GM12878 and the
GTEx LCL track are fetched in the same API call (`ontology_terms=[GM12878_ONTOLOGY,
GTEX_LCL_ONTOLOGY]`) at no extra cost -- GM12878 is the primary comparison target (assay-matched
to GEUVADIS's poly-A selected protocol on both strands); the GTEx LCL track is unstranded
(`strand="."`, single column) and used as a secondary cross-check in Section 7.

**Cost and caching.** `N_GENES x N_INDIVIDUALS x 2 haplotypes` AlphaGenome calls (600 for the
pilot's default sizes), each ~4s -> **allow 40-60 minutes for a cold run.** Every prediction is
disk-cached (`ag_predictions/`), and the final per-individual/gene scalar table is cached to
`predictions.csv` -- reruns after the first are effectively instant.

In [ ]:
def ref_window_fasta(chrom: str, win_start: int, win_end: int, gene_name: str) -> Path:
    # Builds (once, cached) the shared reference-genome window FASTA for a gene, header format
    # "chrom:start-end" (1-based) -- see Section 6 markdown for why the header format matters.
    out_dir = NOTEBOOK_CACHE_DIR / "references" / "windows" / gene_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "ref.window.fa"
    if out_path.exists():
        return out_path
    region = f"{chrom}:{win_start + 1}-{win_end}"
    proc = subprocess.run(
        ["samtools", "faidx", str(REF_FASTA_PATH), region],
        capture_output=True, text=True, check=True,
    )
    seq_only = "".join(l for l in proc.stdout.splitlines() if not l.startswith(">"))
    out_path.write_text(f">{region}\n{seq_only}\n")
    return out_path


def gene_exon_boxes(gene_name: str, chrom: str, win_start: int, win_end: int):
    # MANE Select exon spans in local window coordinates -- same extraction pattern
    # genotype_cnn_alignment_deeplift.ipynb's Section 8 uses (gene_exon_local_boxes).
    interval = genome.Interval(chromosome=chrom, start=win_start, end=win_end)
    target_gene_id = gene_id_map.get(gene_name)
    transcripts = [t for t in transcript_extractor_mane.extract(interval) if t.gene_id == target_gene_id]
    boxes = []
    for t in transcripts:
        for exon in t.exons:
            s, e = max(0, exon.start - win_start), min(win_end - win_start, exon.end - win_start)
            if e > s:
                boxes.append((s, e))
    return sorted(boxes)


AG_PREDICTION_CACHE_DIR = NOTEBOOK_CACHE_DIR / "ag_predictions"
AG_PREDICTION_CACHE_DIR.mkdir(exist_ok=True)


def predict_rna_seq_cached(cache_key: str, sequence: str, interval) -> tuple:
    # Live AlphaGenome RNA_SEQ call for `sequence`, restricted to the two LCL ontology terms,
    # disk-cached by `cache_key`. Talks to dna_client directly (not the shared
    # LiveAlphaGenomePredictor wrapper) because that wrapper's cache only keeps
    # (ontology_curie, strand) per column -- not enough to distinguish GM12878's "polyA plus
    # RNA-seq" columns from its "total RNA-seq" columns, which share the same
    # (ontology_curie, strand) pairs. This notebook needs that distinction to match GEUVADIS's
    # poly-A selected protocol.
    values_path = AG_PREDICTION_CACHE_DIR / f"{cache_key}.npz"
    meta_path = AG_PREDICTION_CACHE_DIR / f"{cache_key}_meta.json"
    if values_path.exists() and meta_path.exists():
        values = np.load(values_path)["values"]
        meta_df = pd.DataFrame(json.loads(meta_path.read_text()))
        return values, meta_df

    output = ag_client.predict_sequence(
        sequence, organism=ORGANISM,
        requested_outputs=[dna_client.OutputType.RNA_SEQ],
        ontology_terms=[GM12878_ONTOLOGY, GTEX_LCL_ONTOLOGY],
        interval=interval,
    )
    values = np.asarray(output.rna_seq.values)
    meta_df = output.rna_seq.metadata[["ontology_curie", "strand", "Assay title", "biosample_name"]].reset_index(drop=True)

    tmp_values_path = values_path.with_suffix(".tmp.npz")
    np.savez_compressed(tmp_values_path, values=values)
    os.replace(tmp_values_path, values_path)
    meta_path.write_text(meta_df.to_json(orient="records"))
    return values, meta_df


def scalar_from_track(values: np.ndarray, meta_df: pd.DataFrame, exon_boxes) -> dict:
    # Sums polyA-plus-RNA-seq signal over the exon union, separately for GM12878 (both strands
    # summed) and the GTEx LCL track (single unstranded column).
    is_polya = meta_df["Assay title"].str.strip() == "polyA plus RNA-seq"

    gm12878_cols = meta_df.index[is_polya & (meta_df["ontology_curie"] == GM12878_ONTOLOGY) & meta_df["strand"].isin(["+", "-"])]
    gtex_lcl_cols = meta_df.index[is_polya & (meta_df["ontology_curie"] == GTEX_LCL_ONTOLOGY)]

    gm12878_signal = values[:, gm12878_cols].sum(axis=1)
    gtex_lcl_signal = values[:, gtex_lcl_cols].sum(axis=1)

    exon_sum = lambda signal: float(sum(signal[s:e].sum() for s, e in exon_boxes))
    return {"gm12878_exon_sum": exon_sum(gm12878_signal), "gtex_lcl_exon_sum": exon_sum(gtex_lcl_signal)}

In [ ]:
PREDICTIONS_CSV = NOTEBOOK_CACHE_DIR / "predictions.csv"

if PREDICTIONS_CSV.exists():
    predictions_df = pd.read_csv(PREDICTIONS_CSV)
    print(f"Loaded cached predictions: {len(predictions_df)} rows from {PREDICTIONS_CSV}")
else:
    rows = []
    n_total = len(pilot_genes) * len(pilot_individuals)
    n_done = 0
    for _, gene_row in pilot_genes.iterrows():
        gene_name, chrom = gene_row["gene_name"], gene_row["chrom"]
        win_start, win_end = int(gene_row["win_start"]), int(gene_row["win_end"])
        ref_fa = ref_window_fasta(chrom, win_start, win_end, gene_name)
        exon_boxes = gene_exon_boxes(gene_name, chrom, win_start, win_end)
        interval = genome.Interval(chromosome=chrom, start=win_start, end=win_end)
        vcf_path = resolve_chrom_vcf(chrom)

        for sample_id in pilot_individuals:
            hap_scalars = []
            for hap in ("H1", "H2"):
                seq, _ = build_haplotype_fasta(
                    sample_id=sample_id, target_name=gene_name, chrom=chrom,
                    start=win_start, end=win_end, ref_window_fa=ref_fa,
                    source_vcf_path=vcf_path, haplotype=hap, cache_dir=NOTEBOOK_CACHE_DIR,
                )
                values, meta_df = predict_rna_seq_cached(
                    cache_key=f"{sample_id}_{gene_name}_{hap}", sequence=seq, interval=interval,
                )
                hap_scalars.append(scalar_from_track(values, meta_df, exon_boxes))

            rows.append({
                "gene_name": gene_name,
                "sample_id": sample_id,
                "gm12878_predicted": np.mean([h["gm12878_exon_sum"] for h in hap_scalars]),
                "gtex_lcl_predicted": np.mean([h["gtex_lcl_exon_sum"] for h in hap_scalars]),
            })
            n_done += 1
            if n_done % 25 == 0 or n_done == n_total:
                print(f"  {n_done}/{n_total} (gene/individual pairs) done -- last: {gene_name}/{sample_id}")

    predictions_df = pd.DataFrame(rows)
    predictions_df.to_csv(PREDICTIONS_CSV, index=False)
    print(f"Saved {len(predictions_df)} predictions to {PREDICTIONS_CSV}")

predictions_df.head()

## Section 7 -- Per-gene, cross-individual correlation (the primary question)

For each of the 10 genes: Spearman and Pearson-on-log1p correlation between AlphaGenome's
predicted signal and GEUVADIS's measured RPKM, **across the 30 individuals**. This asks the
targeted question directly -- does the model's signal move with *real, person-to-person*
expression differences for this gene, not just with the gene's overall expression level (which a
single reference-genome prediction could never speak to at all).

In [ ]:
geuv_by_gene = geuv.set_index("ensembl_base")

def gene_correlations(gene_name: str, ensembl_base: str, pred_col: str) -> dict:
    sub = predictions_df[predictions_df["gene_name"] == gene_name].set_index("sample_id")
    measured = geuv_by_gene.loc[ensembl_base, pilot_individuals].astype(float)
    predicted = sub.loc[pilot_individuals, pred_col].astype(float)
    spearman_rho, spearman_p = scistats.spearmanr(predicted, measured)
    pearson_r, pearson_p = scistats.pearsonr(np.log1p(predicted), np.log1p(measured))
    return {
        "spearman_rho": spearman_rho, "spearman_p": spearman_p,
        "pearson_r_log1p": pearson_r, "pearson_p_log1p": pearson_p,
        "predicted": predicted, "measured": measured,
    }


gm12878_results = {}
gtex_lcl_results = {}
correlation_rows = []
for _, gene_row in pilot_genes.iterrows():
    gene_name, ensembl_base = gene_row["gene_name"], gene_row["ensembl_base"]
    gm12878_results[gene_name] = gene_correlations(gene_name, ensembl_base, "gm12878_predicted")
    gtex_lcl_results[gene_name] = gene_correlations(gene_name, ensembl_base, "gtex_lcl_predicted")
    correlation_rows.append({
        "gene_name": gene_name,
        "gm12878_spearman_rho": gm12878_results[gene_name]["spearman_rho"],
        "gm12878_spearman_p": gm12878_results[gene_name]["spearman_p"],
        "gtex_lcl_spearman_rho": gtex_lcl_results[gene_name]["spearman_rho"],
        "gtex_lcl_spearman_p": gtex_lcl_results[gene_name]["spearman_p"],
    })

correlation_df = pd.DataFrame(correlation_rows).sort_values("gm12878_spearman_rho", ascending=False)
correlation_df

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, (_, row) in zip(axes.flat, pilot_genes.iterrows()):
    gene_name = row["gene_name"]
    r = gm12878_results[gene_name]
    x, y = np.log1p(r["measured"]), np.log1p(r["predicted"])
    ax.scatter(x, y, s=22, alpha=0.75, color="tab:blue")
    if np.ptp(x) > 0:
        slope, intercept = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 50)
        ax.plot(xs, slope * xs + intercept, color="tab:red", linewidth=1.2, linestyle="--")
    ax.set_title(f"{gene_name}\nSpearman rho={r['spearman_rho']:.2f} (p={r['spearman_p']:.3f})", fontsize=10)
    ax.set_xlabel("ln(1 + measured RPKM)", fontsize=8)
    ax.set_ylabel("ln(1 + predicted GM12878 signal)", fontsize=8)
fig.suptitle("Per-gene, cross-individual: predicted (GM12878) vs. measured (GEUVADIS) expression", y=1.02)
fig.tight_layout()
FIG_DIR = NOTEBOOK_CACHE_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
fig.savefig(FIG_DIR / "per_gene_scatter_gm12878.png", dpi=140, bbox_inches="tight")
plt.show()

## Section 8 -- Cross-gene check: does the population mean track *which* genes are more expressed?

A coarser, cheaper question that reuses Section 6's predictions with **zero extra API calls**:
across the 10 genes, does each gene's *population-mean* predicted signal track its *population-
mean* measured RPKM? This is the kind of comparison a single reference-genome AlphaGenome call
per gene could already answer (no personalization needed) -- included here as a sanity check and
a point of contrast with Section 7's harder, individual-level question.

In [ ]:
gene_means = (
    predictions_df.groupby("gene_name")[["gm12878_predicted", "gtex_lcl_predicted"]].mean()
    .join(pilot_genes.set_index("gene_name")[["mean_rpkm"]])
)

cross_gene_rho, cross_gene_p = scistats.spearmanr(gene_means["gm12878_predicted"], gene_means["mean_rpkm"])
print(f"Cross-gene Spearman rho (population-mean predicted vs. population-mean measured): {cross_gene_rho:.3f} (p={cross_gene_p:.3f})")

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(np.log1p(gene_means["mean_rpkm"]), np.log1p(gene_means["gm12878_predicted"]), s=50, color="tab:blue")
for gene_name, row in gene_means.iterrows():
    ax.annotate(gene_name, (np.log1p(row["mean_rpkm"]), np.log1p(row["gm12878_predicted"])), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("ln(1 + population-mean measured RPKM)")
ax.set_ylabel("ln(1 + population-mean predicted GM12878 signal)")
ax.set_title(f"Cross-gene comparison (n={len(gene_means)} genes), Spearman rho={cross_gene_rho:.2f}")
fig.tight_layout()
fig.savefig(FIG_DIR / "cross_gene_scatter.png", dpi=140, bbox_inches="tight")
plt.show()

## Section 9 -- Summary

In [ ]:
summary = {
    "n_genes": len(pilot_genes),
    "n_individuals": len(pilot_individuals),
    "gm12878_median_spearman_rho": float(correlation_df["gm12878_spearman_rho"].median()),
    "gm12878_mean_spearman_rho": float(correlation_df["gm12878_spearman_rho"].mean()),
    "gm12878_n_genes_p_lt_0.05": int((correlation_df["gm12878_spearman_p"] < 0.05).sum()),
    "gtex_lcl_median_spearman_rho": float(correlation_df["gtex_lcl_spearman_rho"].median()),
    "gtex_lcl_mean_spearman_rho": float(correlation_df["gtex_lcl_spearman_rho"].mean()),
    "cross_gene_spearman_rho": float(cross_gene_rho),
}
print(json.dumps(summary, indent=2))
print()
print("Per-gene detail:")
correlation_df

**Reading these numbers.** A per-gene, cross-individual Spearman rho near 0 for most genes (while
the cross-gene rho in Section 8 is clearly positive) would say: AlphaGenome's GM12878/GTEx-LCL
RNA-seq track knows *which genes* are highly expressed in LCLs, but its response to *this
particular cohort's actual genetic variation* at each locus is weak -- i.e. it isn't yet a reliable
per-individual expression predictor for the `genotype_cnn_alignment_deeplift.ipynb` use case, even
though it's a defensible general expression-level feature. A clearly positive per-gene rho for
several genes (even if noisy) would be a more encouraging signal worth following up with more
genes/individuals and, e.g., the DITA-aligned haplotype construction that notebook already uses
(this pilot deliberately used the simpler "raw" consensus sequence, not DITA realignment, to keep
the pipeline self-contained -- see Section 6).

**Caveats specific to this pilot (not fundamental limits).**

- `N_GENES=10`/`N_INDIVIDUALS=30` is a pilot scope, chosen for a first pass with a bounded
  ~40-60 minute API budget -- individual per-gene correlations with n=30 have wide confidence
  intervals; treat single-gene results as suggestive, and the **median across genes** as the more
  reliable summary statistic.
- Gene selection favored **high cross-individual variability** in GEUVADIS itself (Section 4) --
  this maximizes the chance of detecting a real correlation if one exists, but does not
  necessarily represent AlphaGenome's average-case behavior across all genes.
- The exon-sum aggregation is a simple, unweighted proxy for RPKM (no gene-length or library-size
  normalization) -- valid for *within-gene, across-individual* comparisons (Section 7, where gene
  length is constant), less directly comparable *across genes* (Section 8) than RPKM's own
  normalization.